# EfficientNet-B4 — Diabetic Retinopathy Classification
**Kaggle · T4 x2 GPU · EyePACS dataset (35K images · 5 classes)**

| Setting | Value |
|---------|-------|
| Model | EfficientNet-B4 (pretrained ImageNet) |
| Input size | 512 × 512 |
| Epochs | 20 |
| Batch size | 32 |
| Optimizer | Adam lr=1e-4 + CosineAnnealingLR |
| Loss | CrossEntropy with class weights |

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Enable GPU in Settings → Accelerator.')

n_gpus = torch.cuda.device_count()
for i in range(n_gpus):
    name = torch.cuda.get_device_name(i)
    mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'GPU {i}: {name}  ({mem:.1f} GB)')

print(f'\nPyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
device = torch.device('cuda')
print(f'Device  : {device}')

In [ ]:
# ── Cell 2: Install timm ──────────────────────────────────────────────────────
!pip install timm -q
import timm
print(f'timm {timm.__version__} ready')

In [ ]:
# ── Cell 3: Verify dataset paths ──────────────────────────────────────────────
import os, glob

DATA_ROOT = '/kaggle/input/diabetic-retinopathy-detection'
TRAIN_DIR = f'{DATA_ROOT}/train'
TRAIN_CSV = f'{DATA_ROOT}/trainLabels.csv'
OUT_DIR   = '/kaggle/working'

assert os.path.isdir(TRAIN_DIR), f'Train dir not found: {TRAIN_DIR}'
assert os.path.isfile(TRAIN_CSV), f'CSV not found: {TRAIN_CSV}'

images = glob.glob(f'{TRAIN_DIR}/*.jpeg')
print(f'Train images : {len(images):,}')

import pandas as pd
df = pd.read_csv(TRAIN_CSV)
print(f'CSV rows     : {len(df):,}')
print(f'CSV columns  : {list(df.columns)}')
print(f'\nLabel dist:')
print(df['level'].value_counts().sort_index().to_string())
print(f'\nSample rows:')
print(df.head(3).to_string(index=False))

In [ ]:
# ── Cell 4: Config ────────────────────────────────────────────────────────────
# All tuneable knobs in one place

CFG = dict(
    # Paths
    train_dir  = '/kaggle/input/diabetic-retinopathy-detection/train',
    train_csv  = '/kaggle/input/diabetic-retinopathy-detection/trainLabels.csv',
    image_col  = 'image',      # column with image stem (no extension)
    label_col  = 'level',      # column with DR grade 0-4
    img_ext    = '.jpeg',
    out_dir    = '/kaggle/working',

    # Model
    model_name  = 'efficientnet_b4',
    num_classes = 5,
    target_size = 512,
    dropout     = 0.3,

    # Training
    val_split   = 0.15,
    batch_size  = 32,
    num_epochs  = 20,
    lr          = 1e-4,
    num_workers = 4,
    seed        = 42,
)

print('Config:')
for k, v in CFG.items():
    print(f'  {k:<14}: {v}')

In [ ]:
# ── Cell 5: Preprocessing ─────────────────────────────────────────────────────

import cv2
import numpy as np
from PIL import Image


def preprocess_image(image_path: str, target_size: int = 512) -> np.ndarray:
    """Load, resize, and normalise to [0, 1] float32."""
    img = np.array(Image.open(image_path).convert('RGB'))
    img = cv2.resize(img, (target_size, target_size), interpolation=cv2.INTER_AREA)
    return img.astype(np.float32) / 255.0


# Quick sanity-check
sample_path = glob.glob(f"{CFG['train_dir']}/*{CFG['img_ext']}")[0]
sample_arr  = preprocess_image(sample_path, CFG['target_size'])
print(f'Sample shape : {sample_arr.shape}')
print(f'Value range  : [{sample_arr.min():.3f}, {sample_arr.max():.3f}]')

In [ ]:
# ── Cell 6: Dataset class ─────────────────────────────────────────────────────

import torch
from torch.utils.data import Dataset
from pathlib import Path


class DRDataset(Dataset):
    """Diabetic Retinopathy Dataset — works with any CSV (configurable columns)."""

    def __init__(self, image_dir, csv_file,
                 image_col='image', label_col='level',
                 img_ext='.jpeg', transform=None, target_size=512):
        self.image_dir   = Path(image_dir)
        self.transform   = transform
        self.target_size = target_size
        self.img_ext     = img_ext

        df = pd.read_csv(csv_file)
        df.columns = df.columns.str.strip()

        tmp = df[[image_col, label_col]].copy()
        tmp.columns = ['stem', 'label']

        # Only keep rows with an existing image file
        mask = tmp['stem'].apply(
            lambda s: (self.image_dir / f'{s}{img_ext}').exists()
        )
        tmp = tmp[mask].reset_index(drop=True)
        self.stems  = tmp['stem'].tolist()
        self.labels = tmp['label'].tolist()

        skipped = len(df) - len(tmp)
        print(f'DRDataset: {len(tmp):,} images  ({skipped} CSV rows skipped — file not found)')

    def __len__(self):
        return len(self.stems)

    def __getitem__(self, idx):
        path  = self.image_dir / f'{self.stems[idx]}{self.img_ext}'
        image = preprocess_image(str(path), self.target_size)
        image = torch.from_numpy(image).permute(2, 0, 1).float()  # (C, H, W)
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(self.labels[idx], dtype=torch.long)


print('DRDataset class ready.')

In [ ]:
# ── Cell 7: DataLoaders + class weights ───────────────────────────────────────

from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from sklearn.utils.class_weight import compute_class_weight

torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])

# Augmentation applied only to training split
AUGMENT = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
])

# Build full dataset (no augmentation yet)
full_ds = DRDataset(
    image_dir=CFG['train_dir'],
    csv_file=CFG['train_csv'],
    image_col=CFG['image_col'],
    label_col=CFG['label_col'],
    img_ext=CFG['img_ext'],
    target_size=CFG['target_size'],
)

n       = len(full_ds)
n_val   = max(1, int(n * CFG['val_split']))
n_train = n - n_val

generator = torch.Generator().manual_seed(CFG['seed'])
train_ds, val_ds = random_split(full_ds, [n_train, n_val], generator=generator)

# Attach augmentation only to the training split wrapper
train_ds.dataset.transform = AUGMENT

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=CFG['num_workers'], pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], pin_memory=True)

print(f'Train : {n_train:,}  |  Val : {n_val:,}')

# Class weights (no image loading — uses label list directly)
all_labels_np = np.array(full_ds.labels)
present       = np.unique(all_labels_np)
partial_w     = compute_class_weight('balanced', classes=present, y=all_labels_np)
weights_np    = np.ones(CFG['num_classes'], dtype=np.float32)
for cls, w in zip(present, partial_w):
    weights_np[cls] = w

class_weights = torch.tensor(weights_np, dtype=torch.float32, device=device)
print(f'Class weights: {np.round(weights_np, 3)}')

In [ ]:
# ── Cell 8: EfficientNet-B4 model ─────────────────────────────────────────────

import torch.nn as nn


class DRClassifier(nn.Module):
    def __init__(self, num_classes=5, pretrained=True, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b4',
            pretrained=pretrained,
            num_classes=0,   # remove default classifier head
        )
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(self.backbone.num_features, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.backbone(x))


# Use DataParallel to distribute across both T4 GPUs automatically
model = DRClassifier(
    num_classes=CFG['num_classes'],
    pretrained=True,
    dropout=CFG['dropout'],
)
if torch.cuda.device_count() > 1:
    print(f'Using DataParallel across {torch.cuda.device_count()} GPUs')
    model = nn.DataParallel(model)
model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=CFG['lr'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG['num_epochs']
)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total:,}')
print(f'Trainable params: {trainable:,}')

In [ ]:
# ── Cell 9: Training loop (20 epochs) ─────────────────────────────────────────

from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
import json, time


def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = correct = total = 0
    for imgs, labels in tqdm(loader, desc='  train', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += len(labels)
    return total_loss / total, correct / total


@torch.no_grad()
def val_epoch(model, loader, criterion, device, num_classes):
    model.eval()
    total_loss = correct = total = 0
    all_probs, all_labels = [], []
    for imgs, labels in tqdm(loader, desc='  val  ', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss   = criterion(logits, labels)
        all_probs.append(torch.softmax(logits, 1).cpu().numpy())
        all_labels.append(labels.cpu().numpy())
        total_loss += loss.item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += len(labels)
    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    try:
        auroc = roc_auc_score(all_labels, all_probs, multi_class='ovr',
                              average='macro', labels=list(range(num_classes)))
    except ValueError:
        auroc = float('nan')
    return total_loss / total, correct / total, auroc, all_probs, all_labels


# ── Main run ──────────────────────────────────────────────────────────────────
history    = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_auroc': []}
best_auroc = -1.0
best_preds = None
CKPT_PATH  = f"{CFG['out_dir']}/efficientnet_b4_best.pth"

print(f'Training for {CFG["num_epochs"]} epochs  |  train={n_train:,}  val={n_val:,}')
print(f'Checkpoint  → {CKPT_PATH}\n')

for epoch in range(1, CFG['num_epochs'] + 1):
    t0 = time.time()
    print(f'Epoch {epoch}/{CFG["num_epochs"]}')

    tr_loss, tr_acc                          = train_epoch(model, train_loader, criterion, optimizer, device)
    va_loss, va_acc, va_auroc, v_probs, v_lbl = val_epoch(model, val_loader, criterion, device, CFG['num_classes'])
    scheduler.step()
    elapsed = time.time() - t0

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(va_loss)
    history['val_acc'].append(va_acc)
    history['val_auroc'].append(float(va_auroc) if not np.isnan(va_auroc) else 0.0)

    auroc_str = f'{va_auroc:.4f}' if not np.isnan(va_auroc) else 'N/A'
    lr_now    = optimizer.param_groups[0]['lr']
    print(f'  train  loss={tr_loss:.4f}  acc={tr_acc:.3f}')
    print(f'  val    loss={va_loss:.4f}  acc={va_acc:.3f}  AUROC={auroc_str}  lr={lr_now:.2e}  ({elapsed:.0f}s)')

    if not np.isnan(va_auroc) and va_auroc > best_auroc:
        best_auroc = va_auroc
        best_preds = (v_probs, v_lbl)
        torch.save({
            'epoch': epoch,
            'model_state_dict':     model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_auroc': float(va_auroc),
            'val_acc':   float(va_acc),
            'config':    CFG,
        }, CKPT_PATH)
        print(f'  ★ New best checkpoint saved  (AUROC={best_auroc:.4f})')
    print()

# Persist history
history_path = f"{CFG['out_dir']}/training_history.json"
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)

print(f'Best Val AUROC : {best_auroc:.4f}')
print(f'History saved  → {history_path}')

In [ ]:
# ── Cell 10: Training curves ──────────────────────────────────────────────────

import matplotlib.pyplot as plt

epochs = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('EfficientNet-B4 · EyePACS', fontsize=13)

axes[0].plot(epochs, history['train_loss'], label='Train')
axes[0].plot(epochs, history['val_loss'],   label='Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history['train_acc'], label='Train')
axes[1].plot(epochs, history['val_acc'],   label='Val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs, history['val_auroc'], color='steelblue', marker='o', markersize=4)
axes[2].axhline(0.75, color='orange', linestyle='--', alpha=0.7, label='Target 0.75')
axes[2].axhline(0.80, color='red',    linestyle='--', alpha=0.7, label='Target 0.80')
axes[2].set_title('Val AUROC (macro OvR)'); axes[2].set_xlabel('Epoch')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
fig_path = f"{CFG['out_dir']}/training_curves.png"
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {fig_path}')

In [ ]:
# ── Cell 11: Evaluation + confusion matrix ────────────────────────────────────

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Load the best checkpoint for final evaluation
ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Best checkpoint: epoch {ckpt['epoch']}  AUROC={ckpt['val_auroc']:.4f}")

# Re-run validation with best weights
_, _, final_auroc, final_probs, final_labels = val_epoch(
    model, val_loader, criterion, device, CFG['num_classes']
)
final_preds = final_probs.argmax(axis=1)

CLASS_NAMES = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']

print(f'\n=== Final Evaluation (best checkpoint) ===')
print(f'Val AUROC (macro OvR): {final_auroc:.4f}')
print(f'Target range         : 0.75 – 0.80\n')
print(classification_report(final_labels, final_preds, target_names=CLASS_NAMES, digits=3))

# Confusion matrix
fig, ax = plt.subplots(figsize=(7, 6))
cm   = confusion_matrix(final_labels, final_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Confusion Matrix — Val Set  (AUROC={final_auroc:.3f})')
plt.tight_layout()
cm_path = f"{CFG['out_dir']}/confusion_matrix.png"
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {cm_path}')

In [ ]:
# ── Cell 12: Save summary + list /kaggle/working/ ─────────────────────────────

import json

summary = {
    'best_epoch':     int(ckpt['epoch']),
    'best_val_auroc': float(final_auroc),
    'best_val_acc':   float((final_preds == final_labels).mean()),
    'num_classes':    CFG['num_classes'],
    'target_size':    CFG['target_size'],
    'batch_size':     CFG['batch_size'],
    'lr':             CFG['lr'],
    'epochs_run':     len(history['train_loss']),
    'train_images':   n_train,
    'val_images':     n_val,
    'model':          CFG['model_name'],
}

summary_path = f"{CFG['out_dir']}/training_summary.json"
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print('=== Outputs saved to /kaggle/working/ ===')
for fname in sorted(os.listdir(CFG['out_dir'])):
    fpath = os.path.join(CFG['out_dir'], fname)
    size  = os.path.getsize(fpath) / 1e6
    print(f'  {fname:<40}  {size:.1f} MB')

print(f'\nBest Val AUROC : {final_auroc:.4f}')
print('Download the .pth file from the output tab on the right to keep locally.')